In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.python.worker.reuse", "true") \
    .config("spark.ui.port", "4040") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .getOrCreate()

In [5]:
!curl -L https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz -o fhvhv_tripdata_2021-01.csv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

 16  123M   16 20.0M    0     0  13.9M      0  0:00:08  0:00:01  0:00:07 13.9M
 40  123M   40 50.0M    0     0  21.1M      0  0:00:05  0:00:02  0:00:03 32.4M
 56  123M   56 70.0M    0     0  20.2M      0  0:00:06  0:00:03  0:00:03 24.6M
 80  123M   80  100M    0     0  22.9M      0  0:00:05  0:00:04  0:00:01 27.3M
100  123M  100  123M    0     0  24.6M      0  0:00:05  0:00:05 --:--:-- 28.9M


In [10]:
!gunzip fhvhv_tripdata_2021-01.csv.gz

'gunzip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
## count rows
!wc -l fhvhv_tripdata_2021-01.csv

11908469 fhvhv_tripdata_2021-01.csv


In [5]:
## read csv
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [7]:
df.show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
+-----------------+--------------------+-------------------+-------------------+

In [ ]:
## shows everything as string
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

In [10]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [15]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [12]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [13]:
!wc -l head.csv

101 head.csv


In [14]:
import pandas as pd

In [17]:
df_pandas = pd.read_csv('head.csv')

In [18]:
df_pandas.head(10)

,hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
0,HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,NaN
1,HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,NaN
2,HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,NaN
3,HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,NaN
4,HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,NaN
5,HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,NaN
6,HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,NaN
7,HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,NaN
8,HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,NaN
9,HV0003,B02875,2021-01-01 00:40:12,2021-01-01 00:53:31,255,232,NaN


In [19]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [24]:
## create dataframe from pandas
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [13]:
## we will convert LONG type to ingetger since it is more effective
## Long takes 8 bytes while integer take 4 bytes
from pyspark.sql import types

In [26]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [5]:
# create spark dataframe by enforcing schema
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

NameError: name 'schema' is not defined

In [28]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [30]:
df = df.repartition(24)

In [31]:
df.write.parquet('fhvhv/2021/01/')

In [3]:
df = spark.read.parquet('fhvhv/2021/01/')

In [5]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [12]:
df.select('hvfhs_license_num', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

+-----------------+-------------------+-------------------+------------+------------+
|hvfhs_license_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-----------------+-------------------+-------------------+------------+------------+
|           HV0003|2021-01-01 13:31:31|2021-01-01 13:41:50|         225|          37|
|           HV0003|2021-01-01 14:55:01|2021-01-01 15:05:32|         232|         114|
|           HV0003|2021-01-03 00:27:18|2021-01-03 00:36:45|          14|          22|
|           HV0003|2021-01-01 01:34:17|2021-01-01 01:49:01|         172|         221|
|           HV0003|2021-01-01 13:05:55|2021-01-01 13:19:06|         250|         254|
|           HV0003|2021-01-02 09:58:20|2021-01-02 10:35:25|         186|         265|
|           HV0003|2021-01-02 22:49:36|2021-01-02 23:03:54|          17|         225|
|           HV0003|2021-01-02 16:32:53|2021-01-02 17:03:46|          49|          21|
|           HV0003|2021-01-02 06:34:54|2021-01-02 06:5

In [9]:
from pyspark.sql import functions as F

In [9]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date',  'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-01|  2021-01-01|          42|         243|
| 2021-01-01|  2021-01-01|         225|          37|
| 2021-01-02|  2021-01-02|         130|         265|
| 2021-01-02|  2021-01-02|          85|         177|
| 2021-01-01|  2021-01-01|         232|         114|
| 2021-01-03|  2021-01-03|          14|          22|
| 2021-01-01|  2021-01-01|         172|         221|
| 2021-01-01|  2021-01-01|         250|         254|
| 2021-01-02|  2021-01-02|         186|         265|
| 2021-01-01|  2021-01-01|          79|          25|
| 2021-01-02|  2021-01-02|          17|         225|
| 2021-01-02|  2021-01-02|          49|          21|
| 2021-01-02|  2021-01-02|          39|         218|
| 2021-01-03|  2021-01-03|          85|          91|
| 2021-01-03|  2021-01-03|         159|          42|
| 2021-01-01|  2021-01-01|          45|       

In [4]:
def crazy_stuff(base_num):
    # Handle null/None values
    if base_num is None or len(base_num) < 2:
        return None
    
    try:
        num = int(base_num[1:])
        if num % 7 == 0:
            return f's/{num:03x}'
        elif num % 3 == 0:
            return f'a/{num:03x}'
        else:
            return f'e/{num:03x}'
    except (ValueError, IndexError):
        # Handle any parsing errors
        return None

In [5]:
crazy_stuff('B02884')

's/b44'

In [6]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [7]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

Py4JJavaError: An error occurred while calling o66.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (host.docker.internal executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:281)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:309)
	at org.apache.spark.sql.execution.python.BatchEvalPythonEvaluatorFactory.evaluate(BatchEvalPythonExec.scala:95)
	at org.apache.spark.sql.execution.python.EvalPythonEvaluatorFactory$EvalPythonPartitionEvaluator.eval(EvalPythonEvaluatorFactory.scala:113)
	at org.apache.spark.sql.execution.python.EvalPythonExec.$anon